In [6]:
import os
import rasterio
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class BEN14KDataset(Dataset):
    def __init__(self, data_dir, parquet_file, split='train'):
        """
        data_dir: Path to the extracted BEN_14k directory
        parquet_file: Path to metadata.parquet
        split: 'train', 'validation', or 'test'
        """
        self.data_dir = data_dir
        self.split = split
        
        print(f"Loading {split} metadata...")
        df = pd.read_parquet(parquet_file)
        df = df[df['split'] == split]
        
        s1_folder = os.path.join(data_dir, 'BigEarthNet-S1', split)
        s2_folder = os.path.join(data_dir, 'BigEarthNet-S2', split)
        
        # --- BULLETPROOF FIX: Physically check the disk ---
        self.valid_pairs = []
        
        for _, row in df.iterrows():
            # Get possible names from metadata
            patch_id = row.get('patch_id', '')
            s1_name = row.get('s1_name', '')
            s2_name = row.get('s2v1_name', '')
            
            # 1. Check S2 File
            s2_path = None
            if os.path.exists(os.path.join(s2_folder, f"{patch_id}.tif")):
                s2_path = os.path.join(s2_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s2_folder, f"{s2_name}.tif")):
                s2_path = os.path.join(s2_folder, f"{s2_name}.tif")
                
            # 2. Check S1 File
            s1_path = None
            if os.path.exists(os.path.join(s1_folder, f"{patch_id}.tif")):
                s1_path = os.path.join(s1_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s1_folder, f"{s1_name}.tif")):
                s1_path = os.path.join(s1_folder, f"{s1_name}.tif")
                
            # 3. If BOTH files physically exist on your drive, keep them!
            if s1_path and s2_path:
                self.valid_pairs.append((s1_path, s2_path))
                
        # Ab yeh exactly 7180 print karega (kyunki utni hi files hamare folder me thii!)
        print(f"Found {len(self.valid_pairs)} ACTUALLY EXISTING image pairs on disk for {split}.")

    def __len__(self):
        return len(self.valid_pairs)

    def read_tif(self, path):
        with rasterio.open(path) as src:
            return src.read()

    def __getitem__(self, idx):
        # We stored the EXACT valid file paths directly, so no strings can mismatch here
        s1_path, s2_path = self.valid_pairs[idx]
        
        # Read the images
        s1_img = self.read_tif(s1_path) # Sentinel-1 (VV, VH)
        s2_img = self.read_tif(s2_path) # Sentinel-2 (Multispectral)
        
        # --- NORMALIZATION ---
        # SAR: Clip outliers and normalize to [0, 1]
        s1_img = np.clip(s1_img, -25.0, 0.0)
        s1_img = (s1_img - (-25.0)) / 25.0
        
        # Optical: Sentinel-2 reflectance ranges 0 to 10000 -> Normalize to [0, 1]
        s2_img = np.clip(s2_img, 0, 10000) / 10000.0
        
        # Convert to PyTorch tensors
        s1_tensor = torch.tensor(s1_img, dtype=torch.float32)
        s2_tensor = torch.tensor(s2_img, dtype=torch.float32)
        
        return s1_tensor, s2_tensor

# --- Test the DataLoader ---
if __name__ == "__main__":
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
    
    print("Fetching batch...")
    s1_batch, s2_batch = next(iter(dataloader))
    print(f"S1 Batch Shape: {s1_batch.shape}") # Expected: (32, 2, 120, 120)
    print(f"S2 Batch Shape: {s2_batch.shape}") # Expected: (32, 10, 120, 120)
    print("DataLoader is working perfectly! ✅")

Loading train metadata...
Found 7180 ACTUALLY EXISTING image pairs on disk for train.
Fetching batch...
S1 Batch Shape: torch.Size([32, 2, 120, 120])
S2 Batch Shape: torch.Size([32, 10, 120, 120])
DataLoader is working perfectly! ✅


In [ ]:
# Updated class BEN14KDataset():
import os
import rasterio
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

class BEN14KDataset(Dataset):
    def __init__(self, data_dir, parquet_file, split='train'):
        """
        data_dir: Path to the extracted BEN_14k directory
        parquet_file: Path to metadata.parquet
        split: 'train', 'validation', or 'test'
        """
        self.data_dir = data_dir
        self.split = split
        
        print(f"Loading {split} metadata...")
        df = pd.read_parquet(parquet_file)
        df = df[df['split'] == split]
        
        s1_folder = os.path.join(data_dir, 'BigEarthNet-S1', split)
        s2_folder = os.path.join(data_dir, 'BigEarthNet-S2', split)
        
        # --- BULLETPROOF FIX: Physically check the disk ---
        self.valid_pairs = []
        self.patch_ids = []    # Naya: Save patch IDs
        self.labels = []       # Naya: Save ISRO labels for evaluation
        
        for _, row in df.iterrows():
            # Get possible names from metadata
            patch_id = row.get('patch_id', '')
            s1_name = row.get('s1_name', '')
            s2_name = row.get('s2v1_name', '')
            img_labels = row.get('labels', []) # Parquet file se labels uthao
            
            # 1. Check S2 File
            s2_path = None
            if os.path.exists(os.path.join(s2_folder, f"{patch_id}.tif")):
                s2_path = os.path.join(s2_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s2_folder, f"{s2_name}.tif")):
                s2_path = os.path.join(s2_folder, f"{s2_name}.tif")
                
            # 2. Check S1 File
            s1_path = None
            if os.path.exists(os.path.join(s1_folder, f"{patch_id}.tif")):
                s1_path = os.path.join(s1_folder, f"{patch_id}.tif")
            elif os.path.exists(os.path.join(s1_folder, f"{s1_name}.tif")):
                s1_path = os.path.join(s1_folder, f"{s1_name}.tif")
                
            # 3. If BOTH files physically exist on your drive, keep them!
            if s1_path and s2_path:
                self.valid_pairs.append((s1_path, s2_path))
                self.patch_ids.append(patch_id)
                self.labels.append(img_labels)
                
        # Ab yeh exactly 7180 print karega (kyunki utni hi files aapke folder me hain!)
        print(f"Found {len(self.valid_pairs)} ACTUALLY EXISTING image pairs on disk for {split}.")

    def __len__(self):
        return len(self.valid_pairs)

    def read_tif(self, path):
        with rasterio.open(path) as src:
            return src.read()

    def __getitem__(self, idx):
        # We stored the EXACT valid file paths directly, so no strings can mismatch here
        s1_path, s2_path = self.valid_pairs[idx]
        
        # Read the images
        s1_img = self.read_tif(s1_path) # Sentinel-1 (VV, VH)
        s2_img = self.read_tif(s2_path) # Sentinel-2 (Multispectral)
        
        # --- NORMALIZATION ---
        # SAR: Clip outliers and normalize to [0, 1]
        s1_img = np.clip(s1_img, -25.0, 0.0)
        s1_img = (s1_img - (-25.0)) / 25.0
        
        # Optical: Sentinel-2 reflectance ranges 0 to 10000 -> Normalize to [0, 1]
        s2_img = np.clip(s2_img, 0, 10000) / 10000.0
        
        # Convert to PyTorch tensors
        s1_tensor = torch.tensor(s1_img, dtype=torch.float32)
        s2_tensor = torch.tensor(s2_img, dtype=torch.float32)
        
        return s1_tensor, s2_tensor

# --- Test the DataLoader ---
if __name__ == "__main__":
    DATA_DIR = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k"
    PARQUET_FILE = r"C:\Users\ranje\Downloads\ISRO\archive\BEN_14k\metadata.parquet"
    
    dataset = BEN14KDataset(DATA_DIR, PARQUET_FILE, split='train')
    
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
    
    print("Fetching batch...")
    s1_batch, s2_batch = next(iter(dataloader))
    print(f"S1 Batch Shape: {s1_batch.shape}") # Expected: (32, 2, 120, 120)
    print(f"S2 Batch Shape: {s2_batch.shape}") # Expected: (32, 10, 120, 120)
    print("DataLoader is working perfectly! ✅")